# Checking stop-arrival coverage

I am checking whether the current 50 m nearest-stop rule gives enough coverage for the next trip-reconstruction step. The check uses the official route geometry, the processed route-progress rows, and the inferred arrivals. Far-route observations stay out of the arrival counts.

In [1]:
import csv
from collections import Counter, defaultdict
from pathlib import Path
import sys
from statistics import median

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pyarrow.parquet as pq

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from infer_arrivals import infer_arrivals

PROGRESS_PATH = ROOT / "data/processed/route_progress.parquet"
ARRIVALS_PATH = ROOT / "data/processed/stop_arrivals.parquet"
GEOMETRY_PATH = ROOT / "data/processed/route_geometry.csv"
ARTIFACTS = ROOT / "artifacts"

progress = pq.read_table(PROGRESS_PATH).to_pylist()
arrivals = pq.read_table(ARRIVALS_PATH).to_pylist()
with GEOMETRY_PATH.open(newline="", encoding="utf-8") as file:
    geometry = list(csv.DictReader(file))

assert progress and arrivals and geometry
route_keys = sorted({(row["route_id"], row["route_variant_label"]) for row in geometry})
print(f"Loaded {len(progress):,} progress rows, {len(arrivals):,} arrivals, and {len(geometry):,} static stops.")
print("Arrival dates:", sorted({row["inferred_arrival_time"].date().isoformat() for row in arrivals}))

Loaded 84,130 progress rows, 9,930 arrivals, and 193 static stops.
Arrival dates: ['2026-07-30', '2026-07-31', '2026-08-01', '2026-08-02', '2026-08-03', '2026-08-04', '2026-08-05']


## Coverage by route, date, vehicle, and confidence

In [2]:
print("route variant                 passages  dates  vehicles  high  medium  low")
for route_id, variant in route_keys:
    rows = [row for row in arrivals if (row["route_id"], row["route_variant_label"]) == (route_id, variant)]
    confidence = Counter(row["arrival_confidence"] for row in rows)
    dates = {row["inferred_arrival_time"].date() for row in rows}
    vehicles = {row["vehicle_id"] for row in rows if row["vehicle_id"]}
    print(f"{route_id:>4} {variant:<12} {len(rows):>9,}  {len(dates):>5}  {len(vehicles):>8}  {confidence['high']:>4}  {confidence['medium']:>6}  {confidence['low']:>4}")

print("\nPassages by date and route variant")
by_date = Counter((row["inferred_arrival_time"].date().isoformat(), row["route_variant_label"]) for row in arrivals)
for date in sorted({key[0] for key in by_date}):
    values = [f"{variant}={by_date[(date, variant)]}" for _, variant in route_keys]
    print(date, ", ".join(values))

route variant                 passages  dates  vehicles  high  medium  low
1411 ML06UP           4,157      7        49   611    2966   580
1788 274UP            1,478      7       104   215    1128   135
1881 392DOWN          1,387      6        14   197    1071   119
  32 107UP            2,908      7        49   369    2309   230

Passages by date and route variant
2026-07-30 ML06UP=11, 274UP=53, 392DOWN=50, 107UP=32
2026-07-31 ML06UP=869, 274UP=269, 392DOWN=335, 107UP=685
2026-08-01 ML06UP=1054, 274UP=291, 392DOWN=319, 107UP=576
2026-08-02 ML06UP=693, 274UP=275, 392DOWN=247, 107UP=396
2026-08-03 ML06UP=758, 274UP=266, 392DOWN=214, 107UP=621
2026-08-04 ML06UP=640, 274UP=263, 392DOWN=222, 107UP=589
2026-08-05 ML06UP=132, 274UP=61, 392DOWN=0, 107UP=9


In [3]:
arrival_groups = defaultdict(list)
for row in arrivals:
    key = (row["route_id"], row["route_variant_label"], row["stop_id"])
    arrival_groups[key].append(row)

stop_rows = []
for stop in geometry:
    key = (stop["route_id"], stop["route_variant_label"], stop["stop_id"])
    rows = arrival_groups[key]
    confidence = Counter(row["arrival_confidence"] for row in rows)
    stop_rows.append({
        "route_id": stop["route_id"],
        "route_variant_label": stop["route_variant_label"],
        "stop_id": stop["stop_id"],
        "stop_sequence": int(stop["stop_sequence"]),
        "stop_name": stop["stop_name"],
        "passage_count": len(rows),
        "date_count": len({row["inferred_arrival_time"].date() for row in rows}),
        "vehicle_count": len({row["vehicle_id"] for row in rows if row["vehicle_id"]}),
        "high_count": confidence["high"],
        "medium_count": confidence["medium"],
        "low_count": confidence["low"],
    })

for route_id, variant in route_keys:
    rows = [row for row in stop_rows if (row["route_id"], row["route_variant_label"]) == (route_id, variant)]
    observed = [row for row in rows if row["passage_count"]]
    print(
        f"{route_id} {variant}: {len(observed)}/{len(rows)} stops observed, "
        f"passage count median={median(row['passage_count'] for row in rows):.0f}, "
        f"range={min(row['passage_count'] for row in rows)} to {max(row['passage_count'] for row in rows)}, "
        f"vehicle count median={median(row['vehicle_count'] for row in rows):.0f}"
    )

print("\nLowest-coverage stops")
for row in sorted(stop_rows, key=lambda row: (row["passage_count"], row["route_id"], row["stop_sequence"]))[:12]:
    print(
        f"{row['route_id']} {row['route_variant_label']} seq={row['stop_sequence']:>2} "
        f"{row['passage_count']:>3} passages, {row['date_count']} dates, "
        f"{row['vehicle_count']} vehicles, {row['stop_name']}"
    )

vehicle_counts = Counter(row["vehicle_id"] for row in arrivals if row["vehicle_id"])
print(f"\nVehicle coverage: {len(vehicle_counts)} vehicles; top vehicle share={max(vehicle_counts.values()) / len(arrivals):.2%}")
print("Top five vehicle passage counts:", vehicle_counts.most_common(5))
print("Vehicle concentration by route variant")
for route_id, variant in route_keys:
    counts = Counter(row["vehicle_id"] for row in arrivals if (row["route_id"], row["route_variant_label"]) == (route_id, variant))
    print(f"{route_id} {variant}: {len(counts)} vehicles, top vehicle share={max(counts.values()) / sum(counts.values()):.2%}")

1411 ML06UP: 26/26 stops observed, passage count median=157, range=72 to 307, vehicle count median=39
1788 274UP: 47/48 stops observed, passage count median=26, range=0 to 83, vehicle count median=24
1881 392DOWN: 45/46 stops observed, passage count median=26, range=0 to 69, vehicle count median=12
32 107UP: 73/73 stops observed, passage count median=36, range=14 to 150, vehicle count median=22

Lowest-coverage stops
1788 274UP seq=30   0 passages, 0 dates, 0 vehicles, DPS / Police Station Nizamuddin (Lodhi Road)
1881 392DOWN seq= 0   0 passages, 0 dates, 0 vehicles, Noida Sec-62 (Electronic City)
1881 392DOWN seq=24   7 passages, 5 dates, 5 vehicles, DND Border Toll Bridge Plaza
1881 392DOWN seq= 2   8 passages, 5 dates, 8 vehicles, Neelkanth Apptt-Sec-62
1788 274UP seq=13   9 passages, 5 dates, 9 vehicles, Shanti Van Crossing ( Geeta Colony Bridge0
1788 274UP seq=11  10 passages, 4 dates, 10 vehicles, SDM Office
1881 392DOWN seq= 1  10 passages, 5 dates, 6 vehicles, Bajitpur Village


## Stop-sequence coverage plot

Each bar is split into high, medium, and low confidence passages. A missing bar means that no 50 m passage was inferred for that static stop.

In [4]:
colors = {"high": "#2a9d8f", "medium": "#e9c46a", "low": "#e76f51"}
fig, axes = plt.subplots(2, 2, figsize=(14, 9), constrained_layout=True)
for axis, (route_id, variant) in zip(axes.flat, route_keys):
    rows = sorted(
        (row for row in stop_rows if (row["route_id"], row["route_variant_label"]) == (route_id, variant)),
        key=lambda row: row["stop_sequence"],
    )
    sequences = [row["stop_sequence"] for row in rows]
    bottom = [0] * len(rows)
    for label in ("high", "medium", "low"):
        values = [row[f"{label}_count"] for row in rows]
        axis.bar(sequences, values, bottom=bottom, color=colors[label], label=label)
        bottom = [old + value for old, value in zip(bottom, values)]
    axis.set_title(f"{route_id} {variant}")
    axis.set_xlabel("static stop sequence")
    axis.set_ylabel("inferred passages")
    axis.grid(axis="y", alpha=0.2)
axes.flat[0].legend(frameon=False)
ARTIFACTS.mkdir(parents=True, exist_ok=True)
coverage_plot = ARTIFACTS / "arrival_coverage_by_stop.png"
fig.savefig(coverage_plot, dpi=160)
plt.close(fig)
print(f"Wrote {coverage_plot.relative_to(ROOT)}")

Wrote artifacts/arrival_coverage_by_stop.png


## Radius sensitivity

The existing inference function is rerun at 25 m, 40 m, 50 m, and 75 m. I also count the passages added by 75 m relative to the chosen 50 m result.

In [5]:
def arrival_key(row):
    return (
        row["vehicle_id"], row["route_id"], row["route_variant_label"],
        row["trip_id"], row["stop_id"], row["inferred_arrival_time"],
    )

radius_results = {}
base_keys = {arrival_key(row) for row in arrivals}
for radius in (25.0, 40.0, 50.0, 75.0):
    rows = arrivals if radius == 50.0 else infer_arrivals(progress, radius)[0]
    confidence = Counter(row["arrival_confidence"] for row in rows)
    added = [row for row in rows if arrival_key(row) not in base_keys]
    gaps = [row["sampling_gap_seconds"] for row in rows if row["sampling_gap_seconds"] is not None]
    radius_results[radius] = {
        "rows": rows,
        "arrival_count": len(rows),
        "high": confidence["high"],
        "medium": confidence["medium"],
        "low": confidence["low"],
        "low_percent": confidence["low"] / len(rows),
        "vehicle_count": len({row["vehicle_id"] for row in rows if row["vehicle_id"]}),
        "stop_count": len({(row["route_id"], row["stop_id"]) for row in rows}),
        "median_gap": median(gaps),
        "added_count": len(added),
        "added_low": sum(row["arrival_confidence"] == "low" for row in added),
    }

print("radius  passages  high  medium  low  low share  vehicles  stops  median gap  added vs 50m")
for radius, result in radius_results.items():
    print(
        f"{radius:>5.0f}  {result['arrival_count']:>8,}  {result['high']:>4}  {result['medium']:>6}  "
        f"{result['low']:>3}  {result['low_percent']:>8.1%}  {result['vehicle_count']:>8}  "
        f"{result['stop_count']:>5}  {result['median_gap']:>10.0f}s  {result['added_count']:>12,}"
    )

added_75 = radius_results[75.0]["added_count"]
added_75_low = radius_results[75.0]["added_low"]
print(f"75 m additions: {added_75:,} passages, {added_75_low:,} low confidence ({added_75_low / added_75:.1%}).")
assert radius_results[50.0]["arrival_count"] == len(arrivals)

radius  passages  high  medium  low  low share  vehicles  stops  median gap  added vs 50m
   25     5,310   583    4175  552     10.4%       202    191         123s             0
   40     8,229  1079    6271  879     10.7%       212    191         123s             0
   50     9,930  1392    7474  1064     10.7%       216    191         123s             0
   75    13,534  2046   10096  1392     10.3%       219    191         123s         3,604
75 m additions: 3,604 passages, 328 low confidence (9.1%).


## Passage review

This prints 18 inferred passages across all confidence levels and eight far-route observations. The latter are shown as excluded candidates, not as arrivals.

In [6]:
review = []
for confidence in ("high", "medium", "low"):
    candidates = sorted(
        (row for row in arrivals if row["arrival_confidence"] == confidence),
        key=lambda row: (row["route_id"], row["vehicle_id"], row["inferred_arrival_time"], row["stop_sequence"]),
    )
    selected = []
    for route_id, variant in route_keys:
        match = next((row for row in candidates if (row["route_id"], row["route_variant_label"]) == (route_id, variant)), None)
        if match is not None:
            selected.append(match)
    selected.extend(row for row in candidates if row not in selected)
    review.extend(selected[:6])

print("confidence route variant stop seq vehicle time distance gap before after")
for row in review:
    print(
        f"{row['arrival_confidence']:<8} {row['route_id']:>4} {row['route_variant_label']:<10} "
        f"{row['stop_id']:>5} {row['stop_sequence']:>3} {row['vehicle_id']:<12} "
        f"{row['inferred_arrival_time'].isoformat()} {row['minimum_distance_m']:>7.1f}m "
        f"{str(row['sampling_gap_seconds']):>7}s {str(row['has_observation_before']):<5} {str(row['has_observation_after']):<5}"
    )

far_rows = sorted(
    (row for row in progress if row["is_far_from_route"]),
    key=lambda row: (row["route_id"], row["vehicle_id"], row["observation_timestamp"]),
)
far_review = []
for route_id, variant in route_keys:
    matches = [row for row in far_rows if (row["route_id"], row["route_variant_label"]) == (route_id, variant)]
    far_review.extend(matches[:2])
print("\nExcluded far-route candidates")
for row in far_review:
    print(
        f"{row['route_id']:>4} {row['route_variant_label']:<10} {row['vehicle_id']:<12} "
        f"{row['observation_timestamp'].isoformat()} stop_seq={row['nearest_stop_sequence']} "
        f"distance={row['distance_to_nearest_stop_m']:.0f}m excluded=True"
    )

assert len(review) >= 18 and len(review) + len(far_review) >= 20
assert {row["arrival_confidence"] for row in review} == {"high", "medium", "low"}

confidence route variant stop seq vehicle time distance gap before after
high     1411 ML06UP      5478   2 DL1PD8881    2026-07-31T01:20:45+00:00    11.8m   119.0s True  True 
high     1788 274UP        821  15 DL1PD6736    2026-08-04T02:36:10+00:00    18.7m   112.0s True  True 
high     1881 392DOWN     5935  12 DL1PD6792    2026-07-31T02:38:44+00:00     2.6m   112.0s True  True 
high       32 107UP       7614  43 DL1PD4681    2026-07-30T16:57:47+00:00    14.7m   118.0s True  True 
high     1411 ML06UP       328  15 DL1PD8881    2026-07-31T01:48:48+00:00    12.0m   120.0s True  True 
high     1411 ML06UP       279  25 DL1PD8881    2026-07-31T02:35:45+00:00    25.0m   112.0s True  True 
medium   1411 ML06UP      1130   3 DL1PD8881    2026-07-31T01:22:45+00:00    27.6m   121.0s True  True 
medium   1788 274UP       2065   5 DL1PD6736    2026-08-04T02:20:10+00:00    42.7m   112.0s True  True 
medium   1881 392DOWN     5981   5 DL1PD6792    2026-07-31T02:24:45+00:00    28.8m   112.0s Tru

## Decision

I am keeping the 50 m geofence and not adding crossing-time interpolation yet. It reaches 191 of 193 static stops, and the overall arrival table spans all seven collection dates. The 75 m additions are mostly medium confidence rather than low confidence. The archive is not dominated by one vehicle overall, but Route 1881 has only 14 contributing vehicles, so that route is a narrower sample. The limitation is that the inferred time is still the timestamp of a sampled GPS observation, not the exact moment when a bus crossed the stop. Two stops have no usable passages, and coverage is thinner on the lighter route variants, so those gaps need to stay visible in later trip and headway tables.

In [7]:
observed_stop_count = sum(row["passage_count"] > 0 for row in stop_rows)
total_stop_count = len(stop_rows)
far_count = sum(row["is_far_from_route"] for row in progress)
low_count = sum(row["arrival_confidence"] == "low" for row in arrivals)
print(f"50 m coverage: {observed_stop_count}/{total_stop_count} static stops, {len(arrivals):,} passages.")
print(f"Confidence: {low_count:,} low confidence ({low_count / len(arrivals):.1%}); far-route exclusions: {far_count:,}.")
assert observed_stop_count == 191
assert len(arrivals) == 9_930
assert far_count == 7_462

50 m coverage: 191/193 static stops, 9,930 passages.
Confidence: 1,064 low confidence (10.7%); far-route exclusions: 7,462.
